In [1]:
import pandas as pd
import numpy as np
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns

sc.settings.verbosity = 3  # verbosity: errors (0), warnings (1), info (2), hints (3)

In [2]:
unint_soybean = sc.read_h5ad(
    "C:\\Users\\mikep\\git\\Soybean_Single_Cell_drought_heat\\Data\\anndata_export\\adata_rna.h5ad"
)

integrated_soybean = sc.read_h5ad(
    "C:\\Users\\mikep\\git\\Soybean_Single_Cell_drought_heat\\Data\\anndata_export\\adata_integrated.h5ad"
)

In [3]:
unint_soybean

AnnData object with n_obs × n_vars = 30467 × 36042
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample', 'pctCP', 'pctMT', 'nUMI_raw', 'nCount_SCT', 'nFeature_SCT', 'SCT_snn_res.0.3', 'SCT_snn_res.0.4', 'SCT_snn_res.0.5', 'SCT_snn_res.0.6', 'SCT_snn_res.0.7', 'SCT_snn_res.0.8', 'SCT_snn_res.0.9', 'SCT_snn_res.1', 'seurat_clusters', 'treatment', 'libraries', 'replicate', 'source_rds', 'integrated_snn_res.0.5', 'celltype_call', 'cluster_annot'
    var: 'highly_variable', 'in_integrated'
    uns: 'neighbors', 'pca', 'pca_loadings', 'source'
    obsm: 'X_pca', 'X_umap'
    obsp: 'connectivities', 'distances'
    layers: 'counts', None (.X)

In [4]:
integrated_soybean

AnnData object with n_obs × n_vars = 30467 × 3000
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample', 'pctCP', 'pctMT', 'nUMI_raw', 'nCount_SCT', 'nFeature_SCT', 'SCT_snn_res.0.3', 'SCT_snn_res.0.4', 'SCT_snn_res.0.5', 'SCT_snn_res.0.6', 'SCT_snn_res.0.7', 'SCT_snn_res.0.8', 'SCT_snn_res.0.9', 'SCT_snn_res.1', 'seurat_clusters', 'treatment', 'libraries', 'replicate', 'source_rds', 'integrated_snn_res.0.5', 'celltype_call', 'cluster_annot'
    var: 'highly_variable'
    uns: 'pca'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'scale_data', None (.X)

In [5]:
integrated_soybean.layers["scale_data"]

array([[-0.2599243 , -0.14272855, -0.1606693 , ..., -0.03679258,
        -0.06833172, -0.1389724 ],
       [ 2.655449  , -0.14272855, -0.1606693 , ..., -0.03679258,
        -0.06833172, -0.1389724 ],
       [-0.2599243 , -0.14272855, -0.1606693 , ..., -0.03679258,
        -0.06833172, -0.1389724 ],
       ...,
       [ 0.03131191, -0.14272855, -0.1606693 , ..., -0.03679258,
        -0.06833172, -0.1389724 ],
       [-0.19622678, -0.14272855, -0.1606693 , ..., -0.03679258,
        -0.06833172,  0.15402067],
       [-0.21157134, -0.14272855, -0.1606693 , ..., -0.03679258,
        -0.06833172, -0.1389724 ]], shape=(30467, 3000), dtype=float32)

In [6]:
unint_soybean.X = unint_soybean.layers["counts"].copy()
unint_soybean.raw = unint_soybean.copy()

In [7]:
sc.pp.filter_cells(unint_soybean, min_genes=400)
sc.pp.filter_genes(unint_soybean, min_cells=20)

filtered out 68 cells that have less than 400 genes expressed
filtered out 4711 genes that are detected in less than 20 cells


In [8]:
print(f"Number of genes before filtering: {unint_soybean.n_vars}")

max_counts = unint_soybean.X.max(axis=0)
max_counts = max_counts.toarray()
# This explicitly drops genes whose expression is strictly 0 or 1 everywhere - bad for coexpression
## Can drop out those who never reach 3, but it drops about 12k genes. Maybe will do higher and lower confidence coexpression networks?
unint_soybean = unint_soybean[:, max_counts >= 2].copy()

print(f"Number of genes after filtering: {unint_soybean.n_vars}")

Number of genes before filtering: 31331
Number of genes after filtering: 29915


In [9]:
sc.pp.normalize_total(unint_soybean, target_sum=1e4)
sc.pp.log1p(unint_soybean)

normalizing counts per cell
    finished (0:00:02)


In [10]:
sc.pp.highly_variable_genes(unint_soybean, n_top_genes=2500)
sc.tl.pca(unint_soybean, n_comps=50, use_highly_variable=True)

extracting highly variable genes
    finished (0:00:01)
--> added
    'highly_variable', boolean vector (adata.var)
    'means', float vector (adata.var)
    'dispersions', float vector (adata.var)
    'dispersions_norm', float vector (adata.var)
computing PCA
    with n_comps=50


C:\Users\mikep\AppData\Local\Temp\ipykernel_35172\584447359.py:2: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  sc.tl.pca(unint_soybean, n_comps=50, use_highly_variable=True)


    finished (0:00:00)


In [11]:
del unint_soybean.uns["neighbors"]

sc.pp.neighbors(unint_soybean, random_state=476, n_neighbors=20, n_pcs=50)

computing neighbors
    using 'X_pca' with n_pcs = 50
    finished: added to `.uns['neighbors']`
    `.obsp['distances']`, distances for each pair of neighbors
    `.obsp['connectivities']`, weighted adjacency matrix (0:00:26)


In [12]:
unint_soybean.write_h5ad(
    r"C:\Users\mikep\Data\Soybean\Metacells_approaches\Pre_metacells_prepped_data\prepped_data_for_metacells.h5ad"
)

In [13]:
unint_soybean.var

,highly_variable,in_integrated,n_cells,means,dispersions,dispersions_norm
Glyma.01G161700.Wm82.a4.v1,False,False,269,0.084124,2.512328,-0.505831
Glyma.20G213700.Wm82.a4.v1,False,False,24,0.008941,2.623360,0.218367
Glyma.03G158766.Wm82.a4.v1,False,False,976,0.286031,2.560243,-0.269539
Glyma.03G058100.Wm82.a4.v1,False,False,592,0.185253,2.574627,-0.099491
Glyma.09G060300.Wm82.a4.v1,False,False,771,0.228918,2.536523,-0.444006
...,...,...,...,...,...,...
Glyma.04G020400.Wm82.a4.v1,False,False,94,0.026783,2.420883,-1.102271
Glyma.06G292900.Wm82.a4.v1,True,False,1499,0.549400,3.121349,2.832748
Glyma.14G121400.Wm82.a4.v1,False,False,22,0.006621,2.442217,-0.963123
Glyma.03G099950.Wm82.a4.v1,True,False,22,0.009289,2.920231,2.154674
